## Импорты

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder

from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso
from sklearn.linear_model import Ridge

from sklearn.model_selection import GridSearchCV

from sklearn.metrics import r2_score, mean_squared_error as MSE
from math import sqrt

import matplotlib as mlp
from matplotlib import pyplot as plt

# from tqdm import tqdm

## Костыли

In [2]:
def get_region(text):
    try:
        text = text.strip()
    except:
        return 'unknown'
    if isinstance(text, str):
        if ',' in text:
            l = text.split(', ')
        else:
            text = text.replace('  ', ' ')
            l = text.split(' ')
    
        if len(l) == 8:
            if l[5].lower() != 'россия':
                return l[5]
            else:
                return l[4]
        
        elif len(l) == 7:
            if l[4].lower() != 'россия':
                return l[4]
            else:
                return l[3]
                
        elif len(l) == 6 or len(l) == 5:
            if l[3].lower() != 'россия':
                return l[3]
            else:
                return l[2]
                
        elif len(l) == 4:
            if l[2].lower() != 'россия':
                return l[2]
            else:
                return l[1]
    
        elif len(l) == 3:
            if l[2].lower() != 'россия':
                return l[2]
            else:
                return l[1]
    
        elif len(l) == 2:
            if l[1].lower() != 'россия':
                return l[1]
            else:
                return l[0]

        else:
            print(text)
            return text

    else:
        return 'unknown'

In [3]:
def rmse(y_true, y_pred):
    return sqrt(MSE(y_true, y_pred))

## Обработка

In [4]:
# Загрузка данных
train_df = pd.read_csv(r'~/Downloads/train_final.csv')
test_df = pd.read_csv(r'~/Downloads/test.csv')
test_target = pd.read_csv(r'~/Downloads/submit.csv')['target']

In [5]:
test_df = pd.concat([test_df, test_target], axis=1)

In [6]:
# Приводим к строковому типу на всякий случай
train_df['atm_group'] = train_df['atm_group'].astype(str)
test_df['atm_group'] = test_df['atm_group'].astype(str)

# Предобработка столбца atm_group
train_df['atm_group'] = train_df['atm_group'].astype(str).fillna('unknown')
test_df['atm_group'] = test_df['atm_group'].astype(str).fillna('unknown')

# One-Hot Encoding
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoder.fit(train_df[['atm_group']])

# Трансформируем оба набора
train_encoded = encoder.transform(train_df[['atm_group']])
test_encoded = encoder.transform(test_df[['atm_group']])

# Создаем DataFrame с закодированными признаками
train_encoded_df = pd.DataFrame(
    train_encoded, 
    columns=encoder.get_feature_names_out(['atm_group'])
)
test_encoded_df = pd.DataFrame(
    test_encoded, 
    columns=encoder.get_feature_names_out(['atm_group'])
)

# Объединяем с исходными данными
train_final = pd.concat([train_df.drop('atm_group', axis=1), train_encoded_df], axis=1)
test_final = pd.concat([test_df.drop('atm_group', axis=1), test_encoded_df], axis=1)

In [7]:
train_final['address_rus'] = train_final['address_rus'].apply(get_region)
test_final['address_rus'] = test_final['address_rus'].apply(get_region)

59.842861


In [8]:
region_counts_train = train_final['address_rus'].value_counts()
region_counts_test = test_final['address_rus'].value_counts()

# редкие 
train_rare_region = region_counts_train[region_counts_train < 30].index
test_rare_region = region_counts_test[region_counts_test < 30].index 

# Заменяем редкие бренды на Other
train_final['address_rus'] = train_final['address_rus'].apply(lambda x: 'Other' if x in train_rare_region else x)
test_final['address_rus'] = test_final['address_rus'].apply(lambda x: 'Other' if x in test_rare_region else x)

In [9]:
train_final = train_final.drop(['Unnamed: 0', 'address'], axis=1)
test_final = test_final.drop(['Unnamed: 0', 'address'], axis=1)

In [10]:
# One-Hot Encoding
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoder.fit(train_final[['address_rus']])

# Трансформируем оба набора
train_encoded = encoder.transform(train_final[['address_rus']])
test_encoded = encoder.transform(test_final[['address_rus']])

# Создаем DataFrame с закодированными признаками
train_encoded_df = pd.DataFrame(
    train_encoded, 
    columns=encoder.get_feature_names_out(['address_rus'])
)
test_encoded_df = pd.DataFrame(
    test_encoded, 
    columns=encoder.get_feature_names_out(['address_rus'])
)

# Объединяем с исходными данными
train_final = pd.concat([train_final.drop('address_rus', axis=1), train_encoded_df], axis=1)
test_final = pd.concat([test_final.drop('address_rus', axis=1), test_encoded_df], axis=1)

In [11]:
train_final = train_final.drop(columns=['lat', 'long', 'atm_nearby'])
test_final = test_final.drop(columns=['lat', 'long', 'atm_nearby'])

train_final['target'] = train_final['target'].fillna(train_final['target'].median())
test_final['target'] = test_final['target'].fillna(train_final['target'].median())

In [12]:
train_final = test_final.dropna()
test_final = test_final.dropna()

In [13]:
# Подготовка фичей и целевой переменной
X_train = train_final.drop(columns='target')
y_train = train_final['target']

X_test = test_final.drop(columns='target')
y_test = test_final['target']

X_train = X_train.dropna()
X_test = X_test.dropna()

# Выбираем только числовые колонки
numeric_columns = [col for col in X_train.select_dtypes(include=[np.number]).columns.tolist() if col not in ('schools_nearby', 'food_nearby')]
X_train = X_train[numeric_columns]
X_test = X_test[numeric_columns]

In [14]:
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_train_scaled = scaler_X.fit_transform(X_train)
y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1)).ravel()

X_test_scaled = scaler_X.transform(X_test)
y_test_scaled = scaler_y.transform(y_train.values.reshape(-1, 1)).ravel()

## Модель

In [15]:
baseline = LinearRegression()
ridge = Ridge(alpha=0.5)
lasso = Lasso(alpha=0.5)

In [16]:
baseline.fit(X_train_scaled, y_train_scaled)
ridge.fit(X_train_scaled, y_train_scaled)
lasso.fit(X_train_scaled, y_train_scaled)

,alpha,0.5
,fit_intercept,True
,precompute,False
,copy_X,True
,max_iter,1000
,tol,0.0001
,warm_start,False
,positive,False
,random_state,None
,selection,'cyclic'


In [17]:
y_pred_train_baseline = baseline.predict(X_train_scaled)
y_pred_train_ridge = baseline.predict(X_train_scaled)
y_pred_train_lasso = baseline.predict(X_train_scaled)

y_pred_test_baseline = baseline.predict(X_test_scaled)
y_pred_test_ridge = baseline.predict(X_test_scaled)
y_pred_test_lasso = baseline.predict(X_test_scaled)

# Преобразуем обратно в исходный масштаб
y_pred_train_original_baseline = scaler_y.inverse_transform(y_pred_train_baseline.reshape(-1, 1)).ravel()
y_pred_test_original_baseline = scaler_y.inverse_transform(y_pred_test_baseline.reshape(-1, 1)).ravel()

y_pred_train_original_ridge = scaler_y.inverse_transform(y_pred_train_ridge.reshape(-1, 1)).ravel()
y_pred_test_original_ridge = scaler_y.inverse_transform(y_pred_test_ridge.reshape(-1, 1)).ravel()

y_pred_train_original_lasso = scaler_y.inverse_transform(y_pred_train_lasso.reshape(-1, 1)).ravel()
y_pred_test_original_lasso = scaler_y.inverse_transform(y_pred_test_lasso.reshape(-1, 1)).ravel()

In [23]:
print('Train')
print('=' * 30)

print(f'baseline - {rmse(y_train, y_pred_train_original_baseline)}')
print(f'ridge - {rmse(y_train, y_pred_train_original_ridge)}')
print(f'lasso - {rmse(y_train, y_pred_train_original_lasso)}')

print('', end='\n\n')

print('Test')
print('=' * 30)

print(f'baseline - {rmse(y_test, y_pred_test_original_baseline)}')
print(f'ridge - {rmse(y_test, y_pred_test_original_ridge)}')
print(f'lasso - {rmse(y_test, y_pred_test_original_lasso)}')

Train
baseline - 0.05984488074379425
ridge - 0.05984488074379425
lasso - 0.05984488074379425


Test
baseline - 0.05984488074379425
ridge - 0.05984488074379425
lasso - 0.05984488074379425
